# Momentum

In Section 12.4 we reviewed what happens when performing stochastic gradient descent, i.e., when performing optimization where only a noisy variant of the gradient is available.

In particular, we noticed that for noisy gradients we need to be extra cautious when it comes to choosing the learning rate in the face of noise.

If we decrease it too rapidly, convergence stalls.

If we are too lenient, we fail to converge to a good enough solution since noise keeps on driving us away from optimality.

## Basics

In this section, we will explore more effective optimization algorithms, especially for certain types of optimization problems that are common in practice.

## Leaky Averages

The previous section saw us discussing minibatch SGD as a means for accelerating computation. It also had the nice side-effect that averaging gradients reduced the amount of variance. The minibatch stochastic gradient descent can be calculated by:
$$
g_{t, t-1} = \partial_w.\frac{1}{|B_t|}.\sum_{i \in B_t}{f(x_i, w_{t-1})} = \frac{1}{|B_t|}.\sum_{i \in B_t}{h_{i, t-1}}
$$

To keep the notation simple, here we used $h_{i,t-1} = \partial_w.f(x_i, w_{t-1})$ as the stochastic gradient descent for sample $i$ using the weights updated at time $t-1$.It would be nice if we could benefit from the effect of variance reduction even beyond averaging gradients on a minibatch. One option to accomplish this task is to replace the gradient computation by a "leaky average":
$$
v_t = \beta.v_{t-1} + g_{t,t-1}
$$

for some $\beta \in (0,1)$. This effectively replaces the instantaneous gradient by one that is been averaged over multiple past gradients. $v$ is called velocity. It accumulates past gradients similar to how a heavy ball rolling down the objective function landscape integrates over past forces.To see what is happening in more detail let’s expand $v$ recursively into:
$$
v_t = \beta^2.v_{t-2} + \beta.g_{t-1,,t-2} + g_{t,t-1} = ...., = \sum_{k = 0}^{t-1}{\beta^k.g_{t-k,t-k-1}}
$$

Large $\beta$ amounts to a long-range average, whereas small $\beta$ amounts to only a slight correction relative to a gradient method. The new gradient replacement no longer points into the direction of steepest descent on a particular instance any longer but rather in the direction of a weighted average of past gradients. This allows us to realize most of the benefits of averaging over a batch without the cost of actually computing the gradients on it. We will revisit this averaging procedure in more detail later.

The above reasoning formed the basis for what is now known as accelerated gradient methods, such as gradients with momentum.

They enjoy the additional benefit of being much more effective in cases where the optimization problem is ill-conditioned (i.e., where there are some directions where progress is much slower than in others, resembling a narrow canyon). Furthermore, they allow us to average over subsequent gradients to obtain more stable directions of descent. Indeed, the aspect of acceleration even for noise-free convex problems is one of the key reasons why momentum works and why it works so well.

As one would expect, due to its efficacy momentum is a well-studied subject in optimization for deep learning and beyond. See e.g., the beautiful expository article by Goh (2017) for an in-depth analysis and interactive animation. It was proposed by Polyak (1964). Nesterov (2018) has a detailed theoretical discussion in the context of convex optimization. Momentum in deep learning has been known to be beneficial for a long time. See e.g., the discussion by Sutskever et al. (2013) for details.

## An Ill-conditioned Problem

To get a better understanding of the geometric properties of the momentum method we revisit gradient descent, albeit with a significantly less pleasant objective function. Recall that in Section 12.3 we used $f(x) = x_1^2 + 2x_2^2$,i.e., a moderately distorted ellipsoid objective. We distort this function further by stretching it out in the $x_1$ direction via:
$$
f(x) = 0.1x_1^2 + 2x_2^2
$$

As before $f$ has its minimum at $(0,0)$. This function is very flat in the direction of $x_1$. Let’s see what happens when we perform gradient descent as before on this new function. We pick a learning rate of $0.4$

In [ ]:
%matplotlib inline
import torch
from d2l import torch as d2l

eta = 0.4

def f_2d(x1, x2):
    return 0.1 * x1**2 + 2 * x2**2

def gd_2d(x1, x2, s1, s2):
    return (x1 - eta * 0.2 * x1, x2 - eta * 4 * x2, 0, 0)

d2l.show_trace_2d(f_2d, d2l.train_2d(gd_2d))

epoch $20$, x1: $-0.943467$, x2: $-0.000073$

![](image1.png)

By construction, the gradient in the $x_2$ direction is much higher and changes much more rapidly than in the horizontal $x_1$ direction. Thus we are stuck between two undesirable choices:
- if we pick a small learning rate we ensure that the solution does not diverge in the $x_2$ direction but we are saddled with slow convergence in the $x_1$ direction.
- conversely, with a large learning rate we progress rapidly in the $x_1$ direction but diverge in $x_2$.

The example below illustrates what happens even after a slight increase in learning rate from $0.4$ to $0.6$. Convergence in the $x_1$ direction improves but the overall solution quality is much worse.

In [ ]:
eta = 0.6
d2l.show_trace_2d(f_2d, d2l.train_2d(gd_2d))

epoch $20$, x1: $-0.387814$, x2: $-1673.365109$

![](image2.png)

## The Momentum Method

The momentum method allows us to solve the gradient descent problem described above. Looking at the optimization trace above we might intuit that averaging gradients over the past would work well. After all, in the $x_1$ direction this will aggregate well-aligned gradients, thus increasing the distance we cover with every step.

Conversely, in the $x_2$ direction where gradients oscillate, an aggregate gradient will reduce step size due to oscillations that cancel each other out. Using $v_1$ instead of the gradient $g_t$ yields the following update equations:
$$
v_t \leftarrow \beta.v_{t-1} + g_{t,t-1} \\
x_t \leftarrow x_{t-1} - \eta_t.v_t
$$

Note that for $\beta = 0$ we recover regular gradient descent. Before delving deeper into the mathematical properties let’s have a quick look at how the algorithm behaves in practice.

In [ ]:
eta, beta = 0.6, 0.5

def momentum_2d(x1, x2, v1, v2):
    v1 = beta * v1 + 0.2 * x1
    v2 = beta * v2 + 4.0 * x2
    return x1 - eta * v1, x2 - eta * v2, v1, v2

d2l.show_trace_2d(f_2d, d2l.train_2d(momentum_2d))

epoch $20$, x1: $0.007188$, x2: $0.002553$

![](image3.png)

As we can see, even with the same learning rate that we used before, momentum still converges well. Let’s see what happens when we decrease the momentum parameter. Halving it to $\beta = 0.25$ leads to a trajectory that barely converges at all. Nonetheless, it is a lot better than without momentum (when the solution diverges).

In [ ]:
eta, beta = 0.6, 0.25
d2l.show_trace_2d(f_2d, d2l.train_2d(momentum_2d))

epoch $20$, x1: $-0.126340$, x2: $-0.186632$

![](image4.png)

Note that we can combine momentum with stochastic gradient descent and in particular, minibatch stochastic gradient descent. The only change is that in that case we replace the gradients $g_{t,t-1}$ with $g_t$. Last, for convenience we initialize $v_0 = 0$ at time $t = 0$. Let’s look at what leaky averaging actually does to the updates.

## Effective Sample Weight

Recall that $v_t = \sum_{k=0}^{t-1}{\beta^k.g_{t-k, t-k-1}}$.

In the limit the terms add up to $\sum_{k=0}^{\infty}{B^k = \frac{1}{1 - \beta}}$. In other words, rather than taking a step of size $\eta$ in gradient descent or stochastic gradient descent we take a step of size $\frac{\eta}{1 - \beta}$ while at the same time, dealing with a potentially much better behaved descent direction. These are two benefits in one. To illustrate how weighting behaves for different choices of $\beta$ consider the diagram below.

In [ ]:
d2l.set_figsize()

betas = [0.95, 0.9, 0.6, 0]

for beta in betas:
    x = torch.arange(40).detach().numpy()
    d2l.plt.plot(x, beta ** x, label=f'beta = {beta:.2f}')

d2l.plt.xlabel('time')
d2l.plt.legend();

![](image5.png)

## Practical Experiments

Let’s see how momentum works in practice, i.e., when used within the context of a proper optimizer. For this we need a somewhat more scalable implementation.

### Implementation from Scratch

Compared with (minibatch) stochastic gradient descent the momentum method needs to maintain a set of auxiliary variables, i.e., velocity. It has the same shape as the gradients (and variables of the optimization problem). In the implementation below we call these variables states.

In [ ]:
def init_momentum_states(feature_dim):
    v_w = torch.zeros((feature_dim, 1))
    v_b = torch.zeros(1)
    return (v_w, v_b)


def sgd_momentum(params, states, hyperparams):
    for p,v in zip(params, states):
        with torch.no_grad():
            v[:] = hyperparams['momentum'] * v + p.grad
            p[:] -= hyperparams['lr'] * v
        p.grad.data.zero_()

Let’s see how this works in practice.

In [ ]:
data_iter, feature_dim = d2l.get_data_ch11(batch_size=10)

def train_momentum(lr, momentum, num_epochs=2):
    d2l.train_ch11(
        sgd_momentum, 
        init_momentum_states(feature_dim),
        {
            'lr': lr,
            'momentum': momentum
        },
        data_iter,
        feature_dim,
        num_epochs
    )

train_momentum(0.02, 0.5)

loss: $0.245$, $0.153$ sec/epoch

![](image6.png)

When we increase the momentum hyperparameter momentum to 0.9, it amounts to a significantly larger effective sample size of $\frac{1}{1 - 0.9} = 10$. We reduce the learning rate slightly to $0.01$ to keep matters under control.

In [ ]:
train_momentum(0.01, 0.9)

loss: $0.248$, $0.109$ sec/epoch

![](image7.png)

Reducing the learning rate further addresses any issue of non-smooth optimization problems. Setting it to  yields good convergence properties.

In [ ]:
train_momentum(0.005, 0.9)

loss: $0.243$, $0.107$ sec/epoch

![](image8.png)

## Theoretical Analysis

So far the 2D example of $f(x) = 0.1x_1^2 + 2x_2^2$ seemed rather contrived. We will now see that this is actually quite representative of the types of problem one might encounter, at least in the case of minimizing convex quadratic objective functions.

## Quadratic Convex Functions (Bậc hai lồi)

Consider the function:
$$
h(x) = \frac{1}{2}.x^T.Q.x + x^T.c + b
$$

Interpretation:
- $Q$: controls curvature (shape of the bowl)
- $c$: shifts the minimum away from the origin 
- $b$: constant offset (doesn't affect optimization)

if $Q ≻ 0$:
- the function is convex
- there is $\text{one unique minimum}$

This is a general quadratic function. For positive definite matrices $Q ≻ 0$, i.e., for matrices with positive eigenvalues this has a minimizer at $x^* = -Q^{-1}.c$ with minimum value $b - \frac{1}{2}.c^T.Q^{-1}.c$ . Hence we can rewrite $h$ as:
$$
h(x) = \frac{1}{2}(x - Q^{-1}.c)^T.Q(x - Q^{-1}.c) + b - \frac{1}{2}.c^T.Q^{-1}.c
$$

The gradient is given by $\partial_x.h(x) = Q(x - Q^{-1}.c)$. That is, it is given by the distance between $x$ and the minimizer, multiplied by $Q$. Consequently also the velocity is a linear combination of terms $Q(x_t - Q^{-1}.c)$

Since $Q$ is positive definite it can be decomposed into its eigensystem via $Q = O^T.A.O$ for an orthogonal (rotation) matrix $O$ and a diagonal matrix $A$ of positive eigenvalues. This allows us to perform a change of variables from $x$ to $z =^{def} O(x - Q^{-1}.c)$ to obtain a much simplified expression:
$$
h(z) = \frac{1}{2}.z^T.A.z + b^{'}
$$

Here $b^{'} = b - \frac{1}{2}.c^T.Q^{-1}.c$ .Since $O$ is only an orthogonal matrix this does not perturb the gradients in a meaningful way. Expressed in terms of $z$ gradient descent becomes:
$$
z_t = z_{t-1} - A.z_{t-1} = (I - A).z_{t-1}
$$

The important fact in this expression is that gradient descent does not mix between different eigenspaces. That is, when expressed in terms of the eigensystem of $Q$ he optimization problem proceeds in a coordinate-wise manner. This also holds for:
$$
v_t = \beta.v_{t-1} + A.z_{t-1} \\
z_t = z_{t-1} - \eta(\beta.v_{t-1} + A.z_{t-1}) = (I - \beta.A).z_{t-1} - \eta.\beta.v_{t-1}
$$

In doing this we just proved the following theorem: gradient descent with and without momentum for a convex quadratic function decomposes into coordinate-wise optimization in the direction of the eigenvectors of the quadratic matrix.

## Scalar Functions

Given the above result let’s see what happens when we minimize the function $f(x) = \frac{\lambda}{2}x^2$. For gradient descent we have:
$$
x_{t+1} = x_t - \eta.\lambda.x_t = (1 - \eta.\lambda).x_t
$$

Whenever $|1 - \eta.\lambda| < 1$ this optimization converges at an exponential rate since after $t$ steps we have:
$$x_t = (1 - \eta.\lambda)^t.x_0$$

This shows how the rate of convergence improves initially as we increase the learning rate $\eta$ util $\eta.\lambda = 1$. Beyond that things diverge and for $\eta.\lambda > 2$ the optimization problem diverges.

In [ ]:
lambdas = [0.1, 1, 10, 19]
eta = 0.1
d2l.set_figsize((6,4))

for lam in lambdas:
    t = torch.arange(20).detach().numpy()
    d2l.plt.plot(t, (1 - eta * lam) ** t, label=f"lambda = {lam:.2f}")

d2l.plt.xlabel('time')
d2l.plt.legend();

![](image9.png)

To analyze convergence in the case of momentum we begin by rewriting the update equations in terms of two scalars:
- one for $x$
- one for velocity $v$

This yields:
$$
\begin{bmatrix}
v_{t+1} \\
x_{t+1}
\end{bmatrix}
= \begin{bmatrix}
\beta & \lambda \\
-\eta.\beta & (1 - \eta.\lambda)
\end{bmatrix} \begin{bmatrix}v_t \\ x_t\end{bmatrix} = R(\beta, \eta, \lambda)\begin{bmatrix}v_t \\ x_t \end{bmatrix}
$$

We used $R$ to denote the $2 \times 2$ governing convergence behavior. After $t$ steps the initial choice $[v_0, x_0]$ becomes $R(\beta, \eta, \lambda)^t[v_0, x_0]$.Hence, it is up to the eigenvalues of $R$ to determine the speed of convergence. See the Distill post of Goh (2017) for a great animation and Flammarion and Bach (2015) for a detailed analysis. One can show that $0 < \eta.\lambda < 2 + 2\beta$ velocity converges. This is a larger range of feasible parameters when compared to $0 < \eta.\lambda < 2$ for gradient descent. It also suggests that in general large values of $\beta$ are desirable. Further details require a fair amount of technical detail and we suggest that the interested reader consult the original publications.

## Summary

- Momentum replaces gradients with a leaky average over past gradients. This accelerates convergence significantly.
- It is desirable for both noise-free gradient descent and (noisy) stochastic gradient descent.
- Momentum prevents stalling of the optimization process that is much more likely to occur for stochastic gradient descent.
- The effective number of gradients is given by $\frac{1}{1 - \beta}$ due to exponentiated downweighting of past data.
- In the case of convex quadratic problems this can be analyzed explicitly in detail.
- Implementation is quite straightforward but it requires us to store an additional state vector (velocity $v$).